# Axis 1 — Identifiability probes (`30_identifiability`)

**What this is.** The inspectable form of `tests/test_identifiability.py`.
When a covariate coefficient does not recover, there are three possible
explanations: the draw is wrong (code), the model cannot separate the
coefficient from a parameter on the same axis (identifiability), or the
sampler cannot move between the values the posterior allows (mixing). The
probes tell these apart.

**The recipe** (TESTING.md §5): (1) the draw is already unit-tested, so a
failure here is not the draw; (2) fix the partner parameter at truth → the
coefficient recovers; (3) free it and run the chain from **several starting
values** — if the answer depends on where the chain starts, the chain is not
mixing; (4) compare with the **exact posterior**, computed in `_harness` from
the model, never from the sampler.

**How to read it.** For `κ` you will watch a plain two-block Gibbs
alternation (κ given b, b given κ) stay wherever it starts, while the exact
posterior sits near the truth: κ is identified through b's prior, and the
alternation simply cannot travel along the κ/b ridge. The package's own
sampler adds an interweaving step for exactly this reason (1d).


## Setup


In [ ]:
import sys, numpy as np
sys.path.insert(0, '..'); sys.path.insert(0, '../..')
np.set_printoptions(precision=4, suppress=True)
import irtree.gibbs_samplers as gs; import tests._harness as h

def verdict(ok, label):
    print(f"  [{'PASS' if ok else 'FAIL'}]  {label}"); assert ok, label


## 1. `κ` / `b` — identified through b's prior, hard to sample

`b_i` and `Q_i·κ` are both per-item shifts. Without a prior on `b`, only their
sum would be identified. With `b_i ~ N(0, τ²_b)`, κ is the regression of the
item shifts on `Q`, with the `b_i` as residuals: identified, and more weakly
the wider `τ²_b` is (the information comes from I items, not P·I cells).

Sampling is the hard part: κ given b is pinned down by all P·I cells, and b
given κ by P cells each, so a two-block Gibbs sampler takes tiny steps along
the ridge `b − Q·κ = const`. We alternate the two Gibbs draws in isolation
(cheaper than a full fit) to see this.


In [ ]:
def simulate_item_cov(P, I, K, kappa_true, seed):
    rng = np.random.default_rng(seed)
    theta = rng.normal(0,1,P); b = rng.normal(0,0.7,I)
    Q = rng.normal(size=(I,K)); T = np.ones((P,I),bool); ri,ci = np.where(T)
    Z = rng.normal(theta[:,None] - b[None,:] + (Q@kappa_true)[None,:], 1.0)
    return dict(theta=theta,b=b,Q=Q,T=T,ri=ri,ci=ci,Z=Z,P=P,I=I,K=K)

# run_chain draws a coefficient from each node's latent residual with
# gibbs_coef_pooled(kind, nodes, residuals, masks, discriminations, design, tau2, rng);
# kind 'x' = person covariates (beta), 'q' = item covariates (kappa).
def kappa_given_b(d, b, tau2, rng):
    E = d['Z'] - (d['theta'][:,None] - b[None,:])        # latent residual given theta, b
    return gs.gibbs_coef_pooled('q', [0], [E], [d['T']], [None], d['Q'], tau2, rng)

def b_given_kappa(d, kappa, tau2_b, rng):
    Zr = d['Z'] - (d['Q']@kappa)[None,:]
    return gs.gibbs_b(Zr, d['theta'], tau2_b, d['T'], d['ri'], d['ci'], rng)

def alternate(d, tau2_kappa, tau2_b, n_iter, burn, seed, b_fixed=None, kappa0=None):
    rng = np.random.default_rng(seed)
    b = d['b'].copy() if b_fixed is None else b_fixed
    kappa = np.zeros(d['K']) if kappa0 is None else np.array(kappa0, float); draws = []
    for it in range(n_iter):
        if b_fixed is None: b = b_given_kappa(d, kappa, tau2_b, rng)
        kappa = kappa_given_b(d, b, tau2_kappa, rng)
        if it >= burn: draws.append(kappa.copy())
    return np.array(draws).mean(0)


### 1a. `b` fixed at truth → `κ` recovers (proves the draw is fine)


In [ ]:
kt = np.array([0.5, -0.4])
d = simulate_item_cov(1500, 30, 2, kt, seed=1)
est = alternate(d, 4.0, 4.0, 1200, 400, seed=7, b_fixed=d['b'].copy())
print(f'  kappa (b fixed) : {np.round(est,3)}   true {kt}')
verdict(np.abs(est-kt).max() < 0.10, 'kappa recovers with b fixed')


### 1b. `b` free → the answer depends on where the chain starts (a mixing failure)

Run the same alternation from three starting values of κ. If the chain mixed,
all three would agree. Instead each run stays near where it began, so a run
started at 0 reports "κ ≈ 0". The exact posterior (b integrated out
analytically, `_harness.analytic_kappa_marginal`) sits near the truth.


In [ ]:
d = simulate_item_cov(1500, 30, 2, kt, seed=2)
starts = {'0': np.zeros(2), 'truth': kt.copy(), '(2, 2)': np.array([2.0, 2.0])}
ends = {}
for lab, k0 in starts.items():
    ends[lab] = alternate(d, 4.0, 4.0, 1200, 400, seed=7, kappa0=k0)
    print(f'  alternation started at {lab:7s} -> kappa {np.round(ends[lab], 3)}')
mu_exact, V_exact = h.analytic_kappa_marginal(d['Z'], d['theta'], d['T'], d['Q'], tau2_b=4.0, tau2_kappa=4.0)
sd_exact = np.sqrt(np.diag(V_exact))
print(f'  exact posterior mean     {np.round(mu_exact, 3)}  (sd {np.round(sd_exact, 3)})   true {kt}')
spread = max(np.abs(ends[a] - ends[b]).max() for a in ends for b in ends)
verdict(spread > 1.0, 'the alternation ends where it starts (it is not mixing)')
verdict(np.all(np.abs(mu_exact - kt) < 3 * sd_exact), 'the exact posterior is centred near the truth (kappa identified)')


### 1c. Sweep the `b` prior → the alternation catches up as the ridge shortens

For each `τ²_b`, compare the exact posterior mean of κ with the alternation
started at 0. With a wide `b` prior the exact posterior is near the truth but
the alternation barely leaves 0; as `τ²_b` shrinks, `b` can absorb less, the
ridge shortens, and the alternation reaches the exact answer. (An earlier
reading of this sweep, "κ re-emerges as `b` is constrained", was the
alternation catching up, not κ becoming identified.) Reduce `n_iter` for speed.


In [ ]:
d = simulate_item_cov(2000, 40, 2, kt, seed=3)
grid = [4.0, 1.0, 0.25, 0.05]; gaps = []
for tb in grid:
    est = alternate(d, 4.0, tb, 1200, 400, seed=11)
    mu, V = h.analytic_kappa_marginal(d['Z'], d['theta'], d['T'], d['Q'], tau2_b=tb, tau2_kappa=4.0)
    gaps.append(np.abs(est - mu).max())
    print(f'    tau2_b={tb:5.2f}  exact {np.round(mu, 3)}   alternation from 0 {np.round(est, 3)}   gap {gaps[-1]:.3f}')
verdict(gaps[0] > 0.2, 'wide b prior: the alternation is far from the exact posterior (stalled)')
verdict(gaps[-1] < 0.05, 'tight b prior: the alternation reaches the exact posterior')


### 1d. The package's sampler recovers κ with `b` free

`run_chain` follows the κ and b draws with an interweaving step (ASIS, Yu &
Meng 2011) that moves κ and b together along the ridge, so it samples the
posterior the plain alternation could not reach. Here it is fitted to
responses simulated from the same kind of model, with the default
hierarchical prior on `b`. Its target is what 30 items can tell: the
regression of the true item shifts `Q·κ − b` on `Q` (close to, not equal
to, the true κ).


In [ ]:
import warnings, irtree
from irtree.data_prep import expand_data
rng = np.random.default_rng(12); P1, I1 = 1500, 30
Qr = rng.normal(size=(I1, 2)); th1, th2 = rng.normal(0, 1, P1), rng.normal(0, 1, P1)
b1r, b2r = rng.normal(0, 0.7, I1), rng.normal(0, 1, I1)
spec_k = irtree.disengagement_tree(binary=True)
off = (Qr @ kt)[None, :].repeat(P1, 0)
Yk = h.simulate_irtree(spec_k, [dict(theta=th1, b=b1r, offset=off), dict(theta=th2, b=b2r)], P1, I1, rng)
dk = expand_data(Yk, spec_k, categories_start=1, Q=Qr)
nodes_k = [irtree.ProbitRaschNode(item_covariates=True), irtree.ProbitRaschNode()]
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    res_k = irtree.run_chain(dk, nodes_k, irtree.build_default_priors(spec_k, nodes_k),
                             n_iter=3000, seed=1, spec=spec_k, progress=False)
cols = [j for j, n in enumerate(res_k.param_names) if n.startswith('kappa_')]
kd = res_k.chain[:, cols].astype(float)
lo, hi = np.percentile(kd, 2.5, 0), np.percentile(kd, 97.5, 0)
target = np.linalg.lstsq(Qr, Qr @ kt - b1r, rcond=None)[0]
print(f'  kappa posterior mean {np.round(kd.mean(0), 3)}   95% [{np.round(lo, 2)}, {np.round(hi, 2)}]')
print(f'  truth {kt};  regression of the true item shifts on Q {np.round(target, 3)}')
verdict(np.all((lo <= kt) & (kt <= hi)), 'full sampler: the 95% intervals contain the true kappa')


## 2. `β` / `θ` — the identified pair (different axes)

`β` multiplies a *fixed observed* `X_p`; `θ_p` is a *free per-person* effect
on the same axis, but with centred `X` and a proper `θ` prior they are
identified together. So `β` recovers with `θ` fixed **and** free.


In [ ]:
def simulate_person_cov(P, I, K, beta_true, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(P,K)); X = X - X.mean(0)   # centred
    theta = rng.normal(0,1,P); b = rng.normal(0,0.7,I)
    T = np.ones((P,I),bool); ri,ci = np.where(T)
    Z = rng.normal(theta[:,None]-b[None,:]+(X@beta_true)[:,None], 1.0)
    return dict(theta=theta,b=b,X=X,T=T,ri=ri,ci=ci,Z=Z,P=P,I=I,K=K)

def beta_given_theta(d, theta, tau2, rng):
    E = d['Z'] - (theta[:,None] - d['b'][None,:])        # latent residual given theta, b
    return gs.gibbs_coef_pooled('x', [0], [E], [d['T']], [None], d['X'], tau2, rng)

bt = np.array([0.6, -0.5])
d = simulate_person_cov(1500, 20, 2, bt, seed=4)
rng = np.random.default_rng(7)
draws = [beta_given_theta(d, d['theta'], 4.0, rng) for _ in range(1000)]
est = np.mean(draws[300:], axis=0)
print(f'  beta (theta fixed): {np.round(est,3)}   true {bt}')
verdict(np.abs(est-bt).max() < 0.12, 'beta recovers with theta fixed')


In [ ]:
# theta FREE: alternate theta and beta; beta must stay identified
sigma2 = 1.0; Sigma = gs.build_sigma(np.array([sigma2]), np.eye(1))
d = simulate_person_cov(1500, 20, 2, bt, seed=5)
rng = np.random.default_rng(9); beta = np.zeros(2); draws = []
for it in range(1200):
    Zr = d['Z'] - (d['X']@beta)[:,None]
    S,n = gs.compute_suff_stats([Zr],[d['b']],[d['T']],[d['ri']],[d['ci']],[False])
    S,n = gs.reduce_suff_stats_to_dims(S,n,[[0]])
    theta = gs.gibbs_theta_correlated(S,n,Sigma,rng)[:,0]
    beta = beta_given_theta(d, theta, 4.0, rng)
    if it >= 400: draws.append(beta.copy())
est = np.mean(draws, axis=0)
print(f'  beta (theta free) : {np.round(est,3)}   true {bt}')
verdict(np.abs(est-bt).max() < 0.15, 'beta stays identified with free theta')


## Summary

The probes separate three explanations for a coefficient that does not
recover: the draw (ruled out by the unit conditionals), identifiability, and
mixing. For `κ`/`b`, a plain two-block alternation stays wherever it starts,
while the exact posterior computed from the model sits near the truth: κ is
identified through b's prior (weakly when that prior is wide, because the
information comes from the I item locations), and the apparent "collapse" was
a mixing failure. The package's sampler interweaves κ and b and recovers κ
(1d). `β`/`θ` live on different axes and recover either way.
